# 🕸️ Traditional GraphRAG on PostgreSQL

**LLM-extracted entities · communities · local *and* global search — all on your existing
Cloud SQL instance.**

---

This is the *other* branch of GraphRAG from your `Sql_Rag_Know.ipynb`. Same family, opposite
approach to building the graph:

| | Your existing notebook | This notebook |
|---|---|---|
| Where nodes and edges come from | foreign keys + declared document metadata | **an LLM reads every chunk and extracts them** |
| Needs a schema with FKs? | yes, or `MANUAL_EDGES` | **no — works on raw prose** |
| Deterministic? | yes, identical every run | no, varies slightly between runs |
| Build cost | a few SQL queries, free | **one LLM call per chunk** (plus a gleaning pass) |
| Can invent relationships nobody declared? | no | **yes — that is the entire point** |
| Answers "what are the recurring themes?" | no | **yes — global search** |

The pipeline is the one Microsoft's GraphRAG popularised:

```
 INDEX (once, expensive)
   documents → chunks → embeddings
                 ↓
          LLM extraction        "who and what is in this chunk, and how do they relate?"
                 ↓
        entities + relationships + mentions
                 ↓
        community detection      cluster the graph
                 ↓
        LLM community summaries  "what is this cluster about?"

 QUERY (cheap, repeatable)
   LOCAL  question → vector search → entities → 1-hop → their chunks + facts → answer
   GLOBAL question → every community summary → map-reduce → answer
```

**Local search** answers questions about *specific things* — "what happens if Tidewater's
certificate lapses?" **Global search** answers questions about *the corpus as a whole* —
"what are the recurring compliance problems this year?" — which no amount of chunk retrieval
can do, because the answer is not written down in any chunk.

Everything lives in your existing PostgreSQL database in new `gr_*` tables. Nothing touches
`rag_documents`, `rag_chunks` or your business tables — but Section 3 can *read* them so you
do not pay to embed the same corpus twice.

> ⚠️ **This notebook spends money on the index.** Extraction is one Gemini call per chunk.
> 500 chunks ≈ 500 calls. `EXTRACT_MAX_CHUNKS` in Section 1 caps it while you experiment;
> raise it when you are happy with the output.

## 0 — Install

In [ ]:
%pip install -q google-genai psycopg2-binary pgvector tiktoken pandas networkx

## 1 — Configuration

The only cell you normally edit. The database values match your existing notebook, so if that
one connects, this one will too.

In [ ]:
# =========================== Google Cloud / Vertex AI ===========================
PROJECT_ID   = "div-aais-rfpiq-usc1-uat"
LOCATION     = "us-central1"

EXTRACT_MODEL   = "gemini-2.5-flash"   # reads every chunk — must be cheap. This is the bill.
SUMMARY_MODEL   = "gemini-2.5-flash"   # writes community summaries
ANSWER_MODEL    = "gemini-2.5-flash"   # writes the final answer
EMBED_MODEL     = "gemini-embedding-001"
EMBED_DIM       = 1536                 # pgvector's hnsw index refuses >2000 dims

# =========================== PostgreSQL ===========================
DB_CONFIG = {
    "host":     "10.151.179.4",
    "port":     5432,
    "dbname":   "postgres",
    "user":     "postgres",
    "password": "Rfpiq-usc1",          # ⚠️ move this to an env var before this leaves your laptop
    "connect_timeout": 10,
    "application_name": "traditional_graphrag",
}
DB_SCHEMA = "public"

# =========================== Tables (all new; nothing existing is touched) ==========
DOC_T   = "gr_documents"
CHUNK_T = "gr_chunks"
ENT_T   = "gr_entities"
REL_T   = "gr_relations"
MEN_T   = "gr_mentions"
COM_T   = "gr_communities"

# =========================== Chunking ===========================
CHUNK_TOKENS         = 600   # bigger than a pure-RAG chunk ON PURPOSE: extraction needs
                             # enough context to see a relationship, not just a mention.
CHUNK_OVERLAP_TOKENS = 100
MAX_EMBED_TOKENS     = 2000

# =========================== Extraction (the expensive part) ===========================
ENTITY_TYPES = ["ORGANIZATION", "PERSON", "PRODUCT", "FACILITY", "CERTIFICATION",
                "CONTRACT", "RFP", "BID", "EVENT", "LOCATION", "REGULATION"]
EXTRACT_MAX_CHUNKS = 60      # ⬅️ safety cap while experimenting. None = every chunk.
EXTRACT_GLEANINGS  = 1       # extra passes asking "what did you miss?" 0 = off, 1 is plenty.
EXTRACT_WORKERS    = 4       # parallel extraction calls

# =========================== Communities ===========================
COMMUNITY_MIN_SIZE   = 3     # clusters smaller than this are not worth summarising
COMMUNITY_MAX_ENTITIES = 60  # entities described in one summary prompt

# =========================== Retrieval ===========================
VECTOR_CANDIDATES   = 30
TEXT_CANDIDATES     = 30
RRF_K               = 60
LOCAL_SEED_CHUNKS   = 6      # how many hits are used to find starting entities
LOCAL_HOPS          = 1      # traditional GraphRAG local search is 1 hop + descriptions
HUB_DEGREE_MAX      = 80     # never relay through a node this connected
LOCAL_MAX_ENTITIES  = 40
LOCAL_MAX_CHUNKS    = 10
LOCAL_MAX_RELATIONS = 40
MAX_COSINE_DISTANCE = 0.80   # relevance floor - beyond this (and no keyword hit) we refuse

# =========================== Global search ===========================
GLOBAL_BATCH_SIZE   = 5      # community summaries per map call
GLOBAL_MIN_RATING   = 3      # ignore communities the model rated less important than this
GLOBAL_TOP_POINTS   = 20     # key points carried into the reduce step

HNSW_M, HNSW_EF_CONSTRUCTION, HNSW_EF_SEARCH = 16, 64, 100

print(f"Config loaded → {DB_CONFIG['host']}/{DB_CONFIG['dbname']} | "
      f"extract={EXTRACT_MODEL} | embed={EMBED_MODEL}@{EMBED_DIM}d")

## 2 — Database and Vertex AI

`gen()` is the hardened model wrapper: it disables thinking where the model allows it,
detects empty and truncated responses, and retries with a bigger budget instead of letting a
`JSONDecodeError` surface forty lines away from the real cause. Every model call in this
notebook goes through it.

In [ ]:
import contextlib, json, math, random, re, time
from concurrent.futures import ThreadPoolExecutor

import psycopg2, psycopg2.extras
from psycopg2.extras import execute_values
import pandas as pd
import tiktoken
from google import genai
from google.genai.types import GenerateContentConfig, EmbedContentConfig
try:
    from google.genai.types import ThinkingConfig
except ImportError:
    ThinkingConfig = None

# ---------------------------------------------------------------- postgres
_pool = None
def get_conn():
    global _pool
    if _pool is None:
        from psycopg2 import pool as _pgpool
        _pool = _pgpool.SimpleConnectionPool(1, 8, **DB_CONFIG)
    return _pool.getconn()

@contextlib.contextmanager
def db(dict_rows=False, commit=True):
    conn = get_conn()
    try:
        factory = psycopg2.extras.RealDictCursor if dict_rows else None
        with conn.cursor(cursor_factory=factory) as cur:
            yield cur
        conn.commit() if commit else conn.rollback()
    except Exception:
        conn.rollback(); raise
    finally:
        _pool.putconn(conn)

with db() as cur:
    cur.execute("SELECT current_database(), version()")
    _dbname, _ver = cur.fetchone()
print(f"Connected to '{_dbname}' — {_ver.split(',')[0]}")

# ---------------------------------------------------------------- vertex ai
genai_client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)
_enc = tiktoken.get_encoding("cl100k_base")
def n_tokens(t): return len(_enc.encode(t))
def _truncate(t, mx=MAX_EMBED_TOKENS):
    toks = _enc.encode(t)
    return t if len(toks) <= mx else _enc.decode(toks[:mx])

_TRANSIENT = ("429", "RESOURCE_EXHAUSTED", "503", "UNAVAILABLE", "500", "INTERNAL", "DEADLINE")
def _with_retry(fn, attempts=5, base=1.0):
    for i in range(attempts):
        try:
            return fn()
        except Exception as e:
            if i == attempts - 1 or not any(s in str(e) for s in _TRANSIENT):
                raise
            time.sleep(base * (2 ** i) + random.random())

_THINKING_OK = {}
def _thinking_budget_for(model):
    """0 disables thinking. gemini-2.5-pro cannot be disabled — its floor is 128."""
    return 128 if "pro" in model.lower() else 0

def gen(model, prompt, *, max_tokens=4096, json_out=False, temperature=0.0,
        label="model call", _grow=True) -> str:
    """Call Gemini and return TEXT, or raise an error that explains itself.

    gemini-2.5-* models think by default and thinking tokens are drawn from
    max_output_tokens. Too small a budget => empty text => the caller's json.loads throws
    with no hint of the real cause. This function refuses to return an empty string.
    """
    cfg = {"temperature": temperature, "max_output_tokens": int(max_tokens)}
    if json_out:
        cfg["response_mime_type"] = "application/json"
    ladder = []
    if ThinkingConfig is not None and _THINKING_OK.get(model, True):
        ladder.append({**cfg, "thinking_config":
                       ThinkingConfig(thinking_budget=_thinking_budget_for(model))})
    ladder.append(cfg)

    resp, last = None, None
    for c in ladder:
        try:
            resp = _with_retry(lambda c=c: genai_client.models.generate_content(
                model=model, contents=prompt, config=GenerateContentConfig(**c)))
            break
        except Exception as e:
            last = e
            if "thinking_config" in c and ("INVALID_ARGUMENT" in str(e) or "400" in str(e)):
                _THINKING_OK[model] = False
                print(f"ℹ️  {model} rejects thinking_config — dropping it for this session.")
                continue
            raise
    if resp is None:
        raise last

    fr   = str(getattr(resp.candidates[0], "finish_reason", "") or "") if resp.candidates else ""
    text = (resp.text or "").strip()
    th   = getattr(getattr(resp, "usage_metadata", None), "thoughts_token_count", None) or 0
    trunc = fr.upper().endswith("MAX_TOKENS")
    if (not text or trunc) and _grow:
        return gen(model, prompt, max_tokens=int(max_tokens) * 4, json_out=json_out,
                   temperature=temperature, label=label, _grow=False)
    if not text:
        raise RuntimeError(f"{label}: {model} produced no text "
                           f"(finish_reason={fr or '?'}, thinking tokens={th}).")
    return text

def parse_json(text):
    """Models sometimes wrap JSON in ``` fences. Strip, then parse defensively."""
    text = (text or "").strip()
    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z]*\s*|\s*```$", "", text).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        for open_c, close_c in (("[", "]"), ("{", "}")):
            a, b = text.find(open_c), text.rfind(close_c)
            if a != -1 and b > a:
                try:
                    return json.loads(text[a:b + 1])
                except json.JSONDecodeError:
                    continue
        raise

def _l2(v):
    n = math.sqrt(sum(x * x for x in v))
    return [x / n for x in v] if n else v

def embed_texts(texts, task_type, batch_size=16):
    out = []
    for i in range(0, len(texts), batch_size):
        batch = [_truncate(t) for t in texts[i:i + batch_size]]
        r = _with_retry(lambda: genai_client.models.embed_content(
            model=EMBED_MODEL, contents=batch,
            config=EmbedContentConfig(task_type=task_type, output_dimensionality=EMBED_DIM)))
        out.extend(_l2(list(e.values)) for e in r.embeddings)
    return out

def embed_query(q):
    return embed_texts([q], "RETRIEVAL_QUERY")[0]

print("✅ Database + Vertex AI ready.")

## 3 — Schema

Six tables. The first two are ordinary RAG storage; the next four are the graph.

`gr_entities.entity_key` is a **normalised** name (lower-cased, whitespace collapsed). That is
the entity-resolution strategy, and it is deliberately simple: the LLM will emit
`"Tidewater Frozen Holdings"`, `"tidewater frozen holdings"` and `"Tidewater  Frozen Holdings"`
across different chunks, and all three must land on one row. Section 5 has a second, fuzzier
merge pass on top.

Descriptions accumulate: each chunk contributes its own sentence about an entity, and they are
concatenated. Microsoft's implementation LLM-summarises those into one description when they
get long; `describe_entities()` in Section 5 does the same for the worst offenders.

In [ ]:
SCHEMA_DDL = f"""
CREATE EXTENSION IF NOT EXISTS vector;
CREATE EXTENSION IF NOT EXISTS pg_trgm;   -- fuzzy entity matching in Section 5

CREATE TABLE IF NOT EXISTS {DOC_T} (
    doc_id       text PRIMARY KEY,
    title        text NOT NULL,
    content      text NOT NULL,
    metadata     jsonb NOT NULL DEFAULT '{{}}'::jsonb,
    content_hash text,
    indexed_hash text,
    updated_at   timestamptz NOT NULL DEFAULT now()
);

CREATE TABLE IF NOT EXISTS {CHUNK_T} (
    chunk_id     bigserial PRIMARY KEY,
    doc_id       text NOT NULL REFERENCES {DOC_T}(doc_id) ON DELETE CASCADE,
    chunk_index  int  NOT NULL,
    content      text NOT NULL,           -- clean text: shown to the model
    embed_input  text NOT NULL,           -- header + text: embedded and keyword-indexed
    token_count  int,
    embedding    vector({EMBED_DIM}),
    tsv          tsvector GENERATED ALWAYS AS (to_tsvector('english', embed_input)) STORED,
    extracted    boolean NOT NULL DEFAULT false,   -- has the extractor seen this chunk?
    UNIQUE (doc_id, chunk_index)
);

-- ---------------------------------------------------------------- the graph
CREATE TABLE IF NOT EXISTS {ENT_T} (
    entity_key   text PRIMARY KEY,        -- normalised name: 'tidewater frozen holdings'
    name         text NOT NULL,           -- display name as first seen
    entity_type  text NOT NULL,
    description  text NOT NULL DEFAULT '',
    degree       int  NOT NULL DEFAULT 0,
    community    int,                     -- filled in by Section 6
    embedding    vector({EMBED_DIM})       -- optional: lets entities be matched fuzzily
);

CREATE TABLE IF NOT EXISTS {REL_T} (
    rel_id      bigserial PRIMARY KEY,
    src_key     text NOT NULL REFERENCES {ENT_T}(entity_key) ON DELETE CASCADE,
    dst_key     text NOT NULL REFERENCES {ENT_T}(entity_key) ON DELETE CASCADE,
    description text NOT NULL DEFAULT '',
    weight      real NOT NULL DEFAULT 1.0,   -- how strongly the model believes it
    UNIQUE (src_key, dst_key)
);

CREATE TABLE IF NOT EXISTS {MEN_T} (
    entity_key text   NOT NULL REFERENCES {ENT_T}(entity_key) ON DELETE CASCADE,
    chunk_id   bigint NOT NULL REFERENCES {CHUNK_T}(chunk_id) ON DELETE CASCADE,
    doc_id     text   NOT NULL,
    PRIMARY KEY (entity_key, chunk_id)
);

CREATE TABLE IF NOT EXISTS {COM_T} (
    community_id int PRIMARY KEY,
    level        int NOT NULL DEFAULT 0,
    size         int NOT NULL,
    entity_keys  text[] NOT NULL,
    title        text,
    summary      text,
    rating       real,                    -- the model's own 0-10 importance score
    created_at   timestamptz NOT NULL DEFAULT now()
);

CREATE INDEX IF NOT EXISTS {CHUNK_T}_tsv   ON {CHUNK_T} USING gin (tsv);
CREATE INDEX IF NOT EXISTS {CHUNK_T}_doc   ON {CHUNK_T} (doc_id);
CREATE INDEX IF NOT EXISTS {REL_T}_src     ON {REL_T} (src_key);
CREATE INDEX IF NOT EXISTS {REL_T}_dst     ON {REL_T} (dst_key);
CREATE INDEX IF NOT EXISTS {MEN_T}_chunk   ON {MEN_T} (chunk_id);
CREATE INDEX IF NOT EXISTS {ENT_T}_comm    ON {ENT_T} (community);
CREATE INDEX IF NOT EXISTS {ENT_T}_name_tg ON {ENT_T} USING gin (name gin_trgm_ops);
"""

with db() as cur:
    cur.execute(SCHEMA_DDL)
print(f"✅ Schema ready: {DOC_T}, {CHUNK_T}, {ENT_T}, {REL_T}, {MEN_T}, {COM_T}")

## 4 — Chunking and ingest

Two ways to get text in.

**`import_from_rag_tables()`** copies documents out of your existing `rag_documents` and
re-chunks them here. Use this if you already ran the other notebook — you keep one corpus and
one source of truth. It re-embeds, because this notebook chunks at 600 tokens rather than 350;
extraction needs enough context to *see* a relationship, not merely a mention.

**`add_document()`** takes text directly.

`ingest()` is incremental: it hashes content and skips documents whose hash has not moved, so
re-running is cheap and safe.

In [ ]:
HEADER_FIELDS = ["supplier_name", "category", "document_type", "bid_date", "status"]

def contextual_header(doc_id, title, metadata):
    bits = [f"Document: {title}", f"ID: {doc_id}"]
    bits += [f"{f.replace('_', ' ').title()}: {metadata[f]}"
             for f in HEADER_FIELDS if (metadata or {}).get(f)]
    return " | ".join(bits)

def _split(text, max_tokens=None, overlap=None):
    """Paragraph-aware split with overlap, falling back to a hard token split."""
    max_tokens = max_tokens or CHUNK_TOKENS
    overlap    = overlap if overlap is not None else CHUNK_OVERLAP_TOKENS
    paras = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    chunks, cur, cur_tok = [], [], 0
    for p in paras:
        pt = n_tokens(p)
        if pt > max_tokens:                       # a single huge paragraph
            if cur:
                chunks.append("\n\n".join(cur)); cur, cur_tok = [], 0
            toks = _enc.encode(p)
            step = max_tokens - overlap
            for i in range(0, len(toks), step):
                chunks.append(_enc.decode(toks[i:i + max_tokens]))
            continue
        if cur_tok + pt > max_tokens and cur:
            chunks.append("\n\n".join(cur))
            # carry the tail paragraph forward so a fact on a boundary survives whole
            cur, cur_tok = ([cur[-1]], n_tokens(cur[-1])) if overlap else ([], 0)
        cur.append(p); cur_tok += pt
    if cur:
        chunks.append("\n\n".join(cur))
    return [c for c in chunks if c.strip()]

def _hash(s):
    import hashlib
    return hashlib.sha256(s.encode()).hexdigest()

def add_document(doc_id, title, content, metadata=None):
    metadata = metadata or {}
    with db() as cur:
        cur.execute(f"""
            INSERT INTO {DOC_T} (doc_id, title, content, metadata, content_hash)
            VALUES (%s, %s, %s, %s::jsonb, %s)
            ON CONFLICT (doc_id) DO UPDATE
              SET title=EXCLUDED.title, content=EXCLUDED.content,
                  metadata=EXCLUDED.metadata, content_hash=EXCLUDED.content_hash,
                  updated_at=now()
        """, (doc_id, title, content, json.dumps(metadata), _hash(content)))
    return doc_id

def import_from_rag_tables(source_doc_table="rag_documents", limit=None):
    """Copy documents from the other notebook's corpus. Nothing is modified there."""
    try:
        with db(dict_rows=True) as cur:
            cur.execute(f"SELECT doc_id, title, content, metadata FROM {source_doc_table} "
                        + (f"LIMIT {int(limit)}" if limit else ""))
            rows = cur.fetchall()
    except Exception as e:
        print(f"Could not read {source_doc_table}: {type(e).__name__}: {str(e)[:160]}")
        print("Use add_document() instead, or point source_doc_table at your own table.")
        return 0
    for r in rows:
        add_document(r["doc_id"], r["title"], r["content"], r["metadata"] or {})
    print(f"Imported {len(rows)} document(s) from {source_doc_table}.")
    return len(rows)

def ensure_vector_index():
    with db() as cur:
        cur.execute(f"""CREATE INDEX IF NOT EXISTS {CHUNK_T}_vec
                        ON {CHUNK_T} USING hnsw (embedding vector_cosine_ops)
                        WITH (m = {HNSW_M}, ef_construction = {HNSW_EF_CONSTRUCTION})""")

def ingest(force=False, verbose=True):
    """Chunk + embed every document whose content hash has changed."""
    with db(dict_rows=True) as cur:
        cur.execute(f"SELECT doc_id, title, content, metadata, content_hash, indexed_hash "
                    f"FROM {DOC_T}")
        docs = cur.fetchall()
    todo = [d for d in docs if force or d["content_hash"] != d["indexed_hash"]]
    if not todo:
        print(f"Nothing to do — {len(docs)} document(s) already indexed.")
        return 0

    total = 0
    for d in todo:
        header = contextual_header(d["doc_id"], d["title"], d["metadata"] or {})
        pieces = _split(d["content"])
        embed_inputs = [f"{header}\n\n{p}" for p in pieces]
        vecs = embed_texts(embed_inputs, "RETRIEVAL_DOCUMENT")
        with db() as cur:
            cur.execute(f"DELETE FROM {CHUNK_T} WHERE doc_id = %s", (d["doc_id"],))
            execute_values(cur, f"""
                INSERT INTO {CHUNK_T}
                    (doc_id, chunk_index, content, embed_input, token_count, embedding)
                VALUES %s
            """, [(d["doc_id"], i, p, ei, n_tokens(p), v)
                  for i, (p, ei, v) in enumerate(zip(pieces, embed_inputs, vecs))],
               page_size=200)
            cur.execute(f"UPDATE {DOC_T} SET indexed_hash = content_hash WHERE doc_id = %s",
                        (d["doc_id"],))
        total += len(pieces)
        if verbose:
            print(f"   {d['doc_id']:<40} {len(pieces):>3} chunks")
    ensure_vector_index()
    print(f"✅ Ingested {total} chunk(s) across {len(todo)} document(s).")
    return total


# ---- run it -------------------------------------------------------------------------
# import_from_rag_tables()      # ⬅️ uncomment if you already have rag_documents
# add_document("demo-1", "Demo", "Your text here...", {"category": "Demo"})
# ingest()
print("Ingest ready — call import_from_rag_tables() or add_document(), then ingest().")

## 5 — Extraction: the part that makes this *traditional* GraphRAG

This is where an LLM reads every chunk and writes down who and what is in it, and how they
relate. No foreign keys, no metadata, no schema — just prose in, graph out.

Three things in this design are worth understanding, because they are the difference between
a usable graph and a pile of noise.

**Typed entities.** The prompt is given `ENTITY_TYPES` and told to use only those. Without a
closed list the model invents a new type per chunk (`SUPPLIER`, `Supplier`, `VENDOR`,
`COMPANY`…) and nothing groups.

**Gleanings.** After the first pass the model is asked *"what did you miss?"* — with the first
answer in front of it. LLM extraction reliably under-reports on a single pass; this is
Microsoft's trick and it is worth the extra call. `EXTRACT_GLEANINGS = 1` is plenty.

**Relationship strength.** The model rates each relationship 1–10. That becomes the edge weight,
which community detection uses, so a confidently-stated relationship pulls its endpoints into
the same cluster harder than a hedge does.

Extraction is idempotent per chunk (`extracted` flag), so you can stop it, raise
`EXTRACT_MAX_CHUNKS`, and carry on without redoing work.

In [ ]:
EXTRACT_PROMPT = """You are building a knowledge graph from business documents.

From the TEXT below, identify entities and the relationships between them.

ENTITY TYPES — use ONLY these, exactly as written:
{types}

For each entity give:
  - "name": the entity as written, in Title Case. Use the FULL name, not an abbreviation.
  - "type": one of the types above
  - "description": one sentence about this entity, using ONLY what the text says

For each relationship between two entities you listed, give:
  - "source" and "target": entity names exactly as you wrote them above
  - "description": one sentence explaining HOW they are related, from the text
  - "strength": integer 1-10, how strongly and explicitly the text states this

Rules:
- Only entities and relationships the TEXT actually supports. Invent nothing.
- Prefer specific over generic: "Tidewater Frozen Holdings", not "the supplier".
- Skip pure values (dates, amounts, percentages) as entities — put them in descriptions.
- If the text contains nothing extractable, return empty lists.

Return RAW JSON only:
{{"entities": [{{"name": "...", "type": "...", "description": "..."}}],
  "relationships": [{{"source": "...", "target": "...", "description": "...", "strength": 7}}]}}

TEXT:
{text}"""

GLEAN_PROMPT = """Below is a text and the entities and relationships already extracted from it.

MANY WERE MISSED. Add ONLY the ones that are missing. Do not repeat anything already found.
Use the same entity types and the same JSON format. If nothing was missed, return empty lists.

ALREADY FOUND:
{found}

TEXT:
{text}

Return RAW JSON only, same format."""


def _norm(name: str) -> str:
    """Entity resolution, step one: the key everything collapses onto."""
    return re.sub(r"\s+", " ", (name or "").strip().lower())


def _extract_one(chunk):
    """Extract from a single chunk. Returns (entities, relationships) or ([], []) on failure."""
    text = chunk["content"]
    try:
        data = parse_json(gen(EXTRACT_MODEL,
                              EXTRACT_PROMPT.format(types="\n".join(f"  - {t}" for t in ENTITY_TYPES),
                                                    text=text),
                              max_tokens=4096, json_out=True, label="extraction"))
        ents = [e for e in (data.get("entities") or []) if isinstance(e, dict) and e.get("name")]
        rels = [r for r in (data.get("relationships") or []) if isinstance(r, dict)
                and r.get("source") and r.get("target")]
    except Exception as e:
        print(f"  ⚠️  chunk {chunk['chunk_id']}: extraction failed — "
              f"{type(e).__name__}: {str(e)[:120]}")
        return [], []

    # ---- gleaning passes: LLM extraction under-reports on a single look ----
    for _ in range(EXTRACT_GLEANINGS):
        try:
            found = json.dumps({"entities": [e["name"] for e in ents],
                                "relationships": [[r["source"], r["target"]] for r in rels]})
            more = parse_json(gen(EXTRACT_MODEL,
                                  GLEAN_PROMPT.format(found=found, text=text),
                                  max_tokens=3072, json_out=True, label="gleaning"))
            new_e = [e for e in (more.get("entities") or []) if isinstance(e, dict) and e.get("name")]
            new_r = [r for r in (more.get("relationships") or []) if isinstance(r, dict)
                     and r.get("source") and r.get("target")]
            if not new_e and not new_r:
                break
            seen = {_norm(e["name"]) for e in ents}
            ents += [e for e in new_e if _norm(e["name"]) not in seen]
            rels += new_r
        except Exception:
            break                      # a failed gleaning is not worth failing the chunk over
    return ents, rels


def extract_graph(limit=None, verbose=True):
    """Read unextracted chunks, build entities/relationships/mentions. Resumable."""
    limit = EXTRACT_MAX_CHUNKS if limit is None else limit
    with db(dict_rows=True) as cur:
        cur.execute(f"SELECT chunk_id, doc_id, content FROM {CHUNK_T} "
                    f"WHERE NOT extracted ORDER BY chunk_id "
                    + (f"LIMIT {int(limit)}" if limit else ""))
        chunks = [dict(r) for r in cur.fetchall()]
    if not chunks:
        print("No unextracted chunks. (Run ingest() first, or all chunks are done.)")
        return {"entities": 0, "relationships": 0}

    print(f"Extracting from {len(chunks)} chunk(s) with {EXTRACT_MODEL} "
          f"({EXTRACT_GLEANINGS} gleaning pass)…")
    t0 = time.perf_counter()
    with ThreadPoolExecutor(max_workers=EXTRACT_WORKERS) as pool:
        results = list(pool.map(_extract_one, chunks))

    ent_rows, rel_rows, men_rows = {}, {}, set()
    for chunk, (ents, rels) in zip(chunks, results):
        local = {}                                  # display name -> key, for THIS chunk
        for e in ents:
            key = _norm(e["name"])
            if not key:
                continue
            local[e["name"]] = key
            etype = (e.get("type") or "OTHER").strip().upper()
            if etype not in ENTITY_TYPES:
                etype = "OTHER"
            desc = (e.get("description") or "").strip()
            if key in ent_rows:
                prev = ent_rows[key]
                ent_rows[key] = (prev[0], prev[1],
                                 (prev[2] + " " + desc).strip()[:4000])
            else:
                ent_rows[key] = (e["name"].strip()[:400], etype, desc[:4000])
            men_rows.add((key, chunk["chunk_id"], chunk["doc_id"]))
        for r in rels:
            s, d = local.get(r["source"], _norm(r["source"])), local.get(r["target"], _norm(r["target"]))
            if not s or not d or s == d:
                continue
            if s not in ent_rows or d not in ent_rows:
                continue                            # a relationship to an entity never declared
            a, b = (s, d) if s < d else (d, s)       # undirected: one row per pair
            try:
                w = float(r.get("strength") or 5)
            except (TypeError, ValueError):
                w = 5.0
            desc = (r.get("description") or "").strip()
            if (a, b) in rel_rows:
                pd_, pw = rel_rows[(a, b)]
                rel_rows[(a, b)] = ((pd_ + " " + desc).strip()[:4000], max(pw, w))
            else:
                rel_rows[(a, b)] = (desc[:4000], w)

    # ---- write ----
    with db() as cur:
        if ent_rows:
            execute_values(cur, f"""
                INSERT INTO {ENT_T} (entity_key, name, entity_type, description)
                VALUES %s
                ON CONFLICT (entity_key) DO UPDATE SET
                  description = left({ENT_T}.description || ' ' || EXCLUDED.description, 4000)
            """, [(k, v[0], v[1], v[2]) for k, v in ent_rows.items()], page_size=300)
        if rel_rows:
            execute_values(cur, f"""
                INSERT INTO {REL_T} (src_key, dst_key, description, weight)
                SELECT v.s, v.d, v.desc, v.w
                FROM (VALUES %s) AS v(s, d, desc, w)
                WHERE EXISTS (SELECT 1 FROM {ENT_T} e WHERE e.entity_key = v.s)
                  AND EXISTS (SELECT 1 FROM {ENT_T} e WHERE e.entity_key = v.d)
                ON CONFLICT (src_key, dst_key) DO UPDATE SET
                  weight = GREATEST({REL_T}.weight, EXCLUDED.weight),
                  description = left({REL_T}.description || ' ' || EXCLUDED.description, 4000)
            """, [(a, b, v[0], v[1]) for (a, b), v in rel_rows.items()], page_size=300)
        if men_rows:
            execute_values(cur, f"""
                INSERT INTO {MEN_T} (entity_key, chunk_id, doc_id) VALUES %s
                ON CONFLICT DO NOTHING
            """, list(men_rows), page_size=500)
        cur.execute(f"UPDATE {CHUNK_T} SET extracted = true WHERE chunk_id = ANY(%s)",
                    ([c["chunk_id"] for c in chunks],))
    refresh_degrees()
    print(f"✅ {len(ent_rows)} entities, {len(rel_rows)} relationships, {len(men_rows)} mentions "
          f"in {time.perf_counter() - t0:.0f}s")
    with db(dict_rows=True) as cur:
        cur.execute(f"SELECT count(*) n FROM {CHUNK_T} WHERE NOT extracted")
        left = cur.fetchone()["n"]
    if left:
        print(f"   {left} chunk(s) still unextracted — raise EXTRACT_MAX_CHUNKS and re-run.")
    return {"entities": len(ent_rows), "relationships": len(rel_rows)}


def refresh_degrees():
    """Cache each entity's degree — the hub guard reads it on every query."""
    with db() as cur:
        cur.execute(f"""
            UPDATE {ENT_T} e SET degree = COALESCE(d.n, 0)
            FROM (SELECT k, count(*)::int AS n FROM (
                    SELECT src_key AS k FROM {REL_T}
                    UNION ALL SELECT dst_key FROM {REL_T}) u GROUP BY k) d
            WHERE d.k = e.entity_key
        """)
        cur.execute(f"""UPDATE {ENT_T} SET degree = 0
                        WHERE entity_key NOT IN (SELECT src_key FROM {REL_T}
                                                 UNION SELECT dst_key FROM {REL_T})""")

print("Extraction ready — extract_graph()")

### 5b — Entity resolution

`_norm()` already merged case and whitespace variants. This second pass catches the rest:
`"Tidewater Frozen Holdings Inc."` vs `"Tidewater Frozen Holdings"`, using PostgreSQL's
trigram similarity.

It is **opt-in and it prints what it would do before doing it**, because a bad merge is very
hard to undo — two genuinely different entities fused into one node will silently corrupt every
answer that touches them. Read the list before passing `apply=True`.

In [ ]:
def find_duplicate_entities(threshold=0.82, limit=40):
    """Candidate merges, by trigram similarity on the display name."""
    with db(dict_rows=True) as cur:
        cur.execute(f"""
            SELECT a.entity_key AS keep, b.entity_key AS drop_,
                   a.name AS keep_name, b.name AS drop_name,
                   a.degree AS keep_deg, b.degree AS drop_deg,
                   similarity(a.name, b.name) AS sim
            FROM {ENT_T} a JOIN {ENT_T} b
              ON a.entity_key < b.entity_key
             AND a.entity_type = b.entity_type
             AND similarity(a.name, b.name) >= %s
            ORDER BY sim DESC LIMIT %s
        """, (threshold, limit))
        rows = [dict(r) for r in cur.fetchall()]
    for r in rows:
        print(f"  {r['sim']:.2f}  KEEP {r['keep_name'][:34]:<34} (deg {r['keep_deg']:>3})"
              f"   DROP {r['drop_name'][:34]:<34} (deg {r['drop_deg']:>3})")
    if not rows:
        print("  no candidates above the threshold")
    return rows


def merge_entities(pairs, apply=False):
    """Fold drop_ into keep. Re-points relations and mentions, then deletes the duplicate."""
    if not apply:
        print(f"DRY RUN — {len(pairs)} merge(s) would be applied. Pass apply=True to commit.")
        return 0
    n = 0
    for p in pairs:
        keep, drop = p["keep"], p["drop_"]
        with db() as cur:
            cur.execute(f"UPDATE {MEN_T} SET entity_key=%s WHERE entity_key=%s "
                        f"AND chunk_id NOT IN (SELECT chunk_id FROM {MEN_T} WHERE entity_key=%s)",
                        (keep, drop, keep))
            cur.execute(f"DELETE FROM {MEN_T} WHERE entity_key=%s", (drop,))
            for col, other in (("src_key", "dst_key"), ("dst_key", "src_key")):
                cur.execute(f"""UPDATE {REL_T} r SET {col}=%s
                                WHERE {col}=%s AND NOT EXISTS (
                                  SELECT 1 FROM {REL_T} x
                                  WHERE x.{col}=%s AND x.{other}=r.{other})""",
                            (keep, drop, keep))
            cur.execute(f"DELETE FROM {REL_T} WHERE src_key=%s OR dst_key=%s", (drop, drop))
            cur.execute(f"DELETE FROM {REL_T} WHERE src_key = dst_key")
            cur.execute(f"""UPDATE {ENT_T} SET description =
                              left(description || ' ' ||
                                   (SELECT description FROM {ENT_T} WHERE entity_key=%s), 4000)
                            WHERE entity_key=%s""", (drop, keep))
            cur.execute(f"DELETE FROM {ENT_T} WHERE entity_key=%s", (drop,))
        n += 1
    refresh_degrees()
    print(f"✅ merged {n} duplicate entit(y/ies).")
    return n

print("Entity resolution ready — find_duplicate_entities(), then merge_entities(rows, apply=True)")

## 6 — Communities

Global search cannot read every chunk — that is the whole problem it exists to solve. Instead
the graph is **clustered**, and an LLM writes a summary of each cluster. Those summaries are
what global search reads.

Microsoft uses hierarchical Leiden. `networkx`'s greedy modularity gives the same shape of
result without a compiled dependency, and at your corpus size the difference does not show. The
code falls back to connected components if modularity clustering is unavailable.

Each summary comes with the model's own **importance rating** (0–10). `GLOBAL_MIN_RATING`
uses it to skip trivial clusters at query time — which is what keeps global search affordable
once you have hundreds of communities.

In [ ]:
import networkx as nx

def _load_graph():
    with db(dict_rows=True) as cur:
        cur.execute(f"SELECT entity_key, name, entity_type, description, degree FROM {ENT_T}")
        ents = {r["entity_key"]: dict(r) for r in cur.fetchall()}
        cur.execute(f"SELECT src_key, dst_key, description, weight FROM {REL_T}")
        rels = [dict(r) for r in cur.fetchall()]
    G = nx.Graph()
    for k, e in ents.items():
        G.add_node(k, **e)
    for r in rels:
        G.add_edge(r["src_key"], r["dst_key"],
                   weight=float(r["weight"] or 1.0), description=r["description"])
    return G, ents, rels


def detect_communities(min_size=None, verbose=True):
    """Cluster the graph and write the community id back onto each entity."""
    min_size = COMMUNITY_MIN_SIZE if min_size is None else min_size
    G, ents, _ = _load_graph()
    if G.number_of_edges() == 0:
        print("Graph has no edges — run extract_graph() first.")
        return 0
    try:
        groups = nx.community.greedy_modularity_communities(G, weight="weight")
    except Exception as e:
        print(f"  modularity clustering unavailable ({type(e).__name__}); "
              f"falling back to connected components.")
        groups = list(nx.connected_components(G))

    groups = [sorted(g) for g in groups if len(g) >= min_size]
    groups.sort(key=len, reverse=True)

    with db() as cur:
        cur.execute(f"DELETE FROM {COM_T}")
        cur.execute(f"UPDATE {ENT_T} SET community = NULL")
        for cid, members in enumerate(groups):
            cur.execute(f"INSERT INTO {COM_T} (community_id, level, size, entity_keys) "
                        f"VALUES (%s, 0, %s, %s)", (cid, len(members), members))
            cur.execute(f"UPDATE {ENT_T} SET community=%s WHERE entity_key = ANY(%s)",
                        (cid, members))
    if verbose:
        print(f"{len(groups)} communities (min size {min_size}):")
        for cid, m in enumerate(groups[:10]):
            names = ", ".join(ents[k]["name"][:22] for k in m[:5])
            print(f"   #{cid:<3} {len(m):>3} entities   {names}"
                  + (" …" if len(m) > 5 else ""))
    return len(groups)


COMMUNITY_PROMPT = """You are summarising one cluster of a business knowledge graph.

Write a report on this community: what it is about, who and what is in it, and what the
relationships between them mean for the business.

ENTITIES:
{entities}

RELATIONSHIPS:
{relationships}

Return RAW JSON only:
{{"title": "<short specific name for this community, 3-8 words>",
  "summary": "<200-350 words. Lead with what this community IS. Then the important
              entities and what connects them. Then anything notable: risks, dependencies,
              conflicts, gaps. Use ONLY the information above.>",
  "rating": <0-10, how important this community is for understanding the business overall>}}"""


def summarize_communities(only_missing=True, verbose=True):
    """One LLM call per community. This is what global search reads."""
    G, ents, _ = _load_graph()
    with db(dict_rows=True) as cur:
        cur.execute(f"SELECT community_id, entity_keys, summary FROM {COM_T} ORDER BY size DESC")
        comms = [dict(r) for r in cur.fetchall()]
    todo = [c for c in comms if not (only_missing and c["summary"])]
    if not todo:
        print("All communities already summarised.")
        return 0

    def _one(c):
        keys = c["entity_keys"][:COMMUNITY_MAX_ENTITIES]
        ent_block = "\n".join(
            f"  - {ents[k]['name']} ({ents[k]['entity_type']}): {ents[k]['description'][:300]}"
            for k in keys if k in ents)
        rel_block = "\n".join(
            f"  - {ents[a]['name']} ↔ {ents[b]['name']}: {d.get('description','')[:220]}"
            for a, b, d in G.subgraph(keys).edges(data=True)
            if a in ents and b in ents)[:12000]
        try:
            data = parse_json(gen(SUMMARY_MODEL,
                                  COMMUNITY_PROMPT.format(entities=ent_block,
                                                          relationships=rel_block or "  (none)"),
                                  max_tokens=4096, json_out=True, label="community summary"))
            return c["community_id"], data.get("title", ""), data.get("summary", ""), \
                   float(data.get("rating") or 5)
        except Exception as e:
            print(f"  ⚠️  community {c['community_id']}: {type(e).__name__}: {str(e)[:110]}")
            return None

    with ThreadPoolExecutor(max_workers=EXTRACT_WORKERS) as pool:
        out = [r for r in pool.map(_one, todo) if r]

    with db() as cur:
        for cid, title, summary, rating in out:
            cur.execute(f"UPDATE {COM_T} SET title=%s, summary=%s, rating=%s "
                        f"WHERE community_id=%s", (title, summary, rating, cid))
    if verbose:
        for cid, title, _, rating in sorted(out, key=lambda x: -x[3])[:10]:
            print(f"   #{cid:<3} rating {rating:>4.1f}   {title}")
    print(f"✅ Summarised {len(out)} community(ies).")
    return len(out)

print("Communities ready — detect_communities(), then summarize_communities()")

## 7 — Local search

For questions about *specific things*. This is the classic loop, and it is the same shape as
Section G of your other notebook — the difference is that the entities it walks were extracted
by an LLM rather than derived from foreign keys.

The model is given four kinds of context, and the mix matters: **entity descriptions** (what
these things are), **relationship descriptions** (how they connect — this is what plain RAG
cannot produce), **community summaries** for the entities involved (the wider context), and
**source chunks** (the verbatim evidence it must cite).

In [ ]:
REFUSAL = "I don't have that in the indexed documents."

def retrieve(question, qv=None, top_k=None, seed_only=False):
    """Hybrid vector + keyword retrieval, fused with Reciprocal Rank Fusion."""
    qv = qv or embed_query(question)
    top_k = top_k or LOCAL_MAX_CHUNKS
    with db(dict_rows=True) as cur:
        cur.execute(f"SET LOCAL hnsw.ef_search = {HNSW_EF_SEARCH}")
        cur.execute(f"""
            WITH v AS (
              SELECT chunk_id, row_number() OVER (ORDER BY embedding <=> %(qv)s::vector) AS rnk,
                     (embedding <=> %(qv)s::vector) AS dist
              FROM {CHUNK_T} WHERE embedding IS NOT NULL
              ORDER BY embedding <=> %(qv)s::vector LIMIT {VECTOR_CANDIDATES}
            ),
            t AS (
              SELECT chunk_id, row_number() OVER (
                       ORDER BY ts_rank_cd(tsv, plainto_tsquery('english', %(q)s)) DESC) AS rnk
              FROM {CHUNK_T} WHERE tsv @@ plainto_tsquery('english', %(q)s)
              LIMIT {TEXT_CANDIDATES}
            ),
            fused AS (
              SELECT COALESCE(v.chunk_id, t.chunk_id) AS chunk_id,
                     COALESCE(1.0/({RRF_K} + v.rnk), 0) + COALESCE(1.0/({RRF_K} + t.rnk), 0) AS score,
                     v.dist, t.rnk AS txt_rank
              FROM v FULL OUTER JOIN t ON v.chunk_id = t.chunk_id
            )
            SELECT f.chunk_id, f.score, COALESCE(f.dist, 1.0) AS cosine_distance, f.txt_rank,
                   c.doc_id, c.chunk_index, c.content, d.title
            FROM fused f
            JOIN {CHUNK_T} c ON c.chunk_id = f.chunk_id
            JOIN {DOC_T}   d ON d.doc_id   = c.doc_id
            ORDER BY f.score DESC LIMIT %(k)s
        """, {"qv": qv, "q": question, "k": top_k})
        return [dict(r) for r in cur.fetchall()]


def entities_in_chunks(chunk_ids):
    if not chunk_ids:
        return []
    with db(dict_rows=True) as cur:
        cur.execute(f"""
            SELECT m.entity_key, e.name, e.entity_type, e.description, e.degree, e.community,
                   count(*) AS hits
            FROM {MEN_T} m JOIN {ENT_T} e ON e.entity_key = m.entity_key
            WHERE m.chunk_id = ANY(%s)
            GROUP BY m.entity_key, e.name, e.entity_type, e.description, e.degree, e.community
            ORDER BY hits DESC, e.degree DESC
        """, (list(chunk_ids),))
        return [dict(r) for r in cur.fetchall()]


def expand(keys, hops=None, hub_max=None, limit=None):
    """Walk outward. A hub can be an ANSWER but is never relayed THROUGH."""
    hops    = LOCAL_HOPS if hops is None else hops
    hub_max = HUB_DEGREE_MAX if hub_max is None else hub_max
    limit   = LOCAL_MAX_ENTITIES if limit is None else limit
    seen, frontier = {}, list(dict.fromkeys(keys))
    if not frontier:
        return []
    with db(dict_rows=True) as cur:
        cur.execute(f"""SELECT entity_key, name, entity_type, description, degree, community
                        FROM {ENT_T} WHERE entity_key = ANY(%s)""", (frontier,))
        for r in cur.fetchall():
            seen[r["entity_key"]] = dict(r, hop=0)
        for hop in range(1, hops + 1):
            if not frontier:
                break
            cur.execute(f"""
                SELECT DISTINCT e.entity_key, e.name, e.entity_type, e.description,
                       e.degree, e.community
                FROM {REL_T} r
                JOIN {ENT_T} src ON src.entity_key = CASE WHEN r.src_key = ANY(%(f)s)
                                                          THEN r.src_key ELSE r.dst_key END
                JOIN {ENT_T} e   ON e.entity_key   = CASE WHEN r.src_key = ANY(%(f)s)
                                                          THEN r.dst_key ELSE r.src_key END
                WHERE (r.src_key = ANY(%(f)s) OR r.dst_key = ANY(%(f)s))
                  AND src.degree <= %(hub)s
            """, {"f": frontier, "hub": hub_max})
            nxt = []
            for r in cur.fetchall():
                if r["entity_key"] not in seen:
                    seen[r["entity_key"]] = dict(r, hop=hop)
                    nxt.append(r["entity_key"])
            frontier = nxt
    out = sorted(seen.values(), key=lambda r: (r["hop"], -(r["degree"] or 0)))
    return out[:limit]


def relations_between(keys, limit=None):
    if not keys:
        return []
    with db(dict_rows=True) as cur:
        cur.execute(f"""
            SELECT a.name AS src, b.name AS dst, r.description, r.weight
            FROM {REL_T} r
            JOIN {ENT_T} a ON a.entity_key = r.src_key
            JOIN {ENT_T} b ON b.entity_key = r.dst_key
            WHERE r.src_key = ANY(%s) AND r.dst_key = ANY(%s)
            ORDER BY r.weight DESC LIMIT %s
        """, (list(keys), list(keys), LOCAL_MAX_RELATIONS if limit is None else limit))
        return [dict(r) for r in cur.fetchall()]


def chunks_for_entities(expanded, exclude=(), limit=None):
    """Back out of the graph into text, weighted by hop distance."""
    if not expanded:
        return []
    keys = [e["entity_key"] for e in expanded]
    ws   = [1.0 / (1 + e.get("hop", 0)) ** 2 for e in expanded]
    with db(dict_rows=True) as cur:
        cur.execute(f"""
            WITH w(entity_key, weight) AS (SELECT * FROM unnest(%(k)s::text[], %(w)s::float8[])),
            scored AS (
              SELECT c.chunk_id, c.doc_id, c.chunk_index, c.content, d.title,
                     count(DISTINCT m.entity_key) AS ent_hits, sum(w.weight) AS graph_score
              FROM {MEN_T} m
              JOIN w ON w.entity_key = m.entity_key
              JOIN {CHUNK_T} c ON c.chunk_id = m.chunk_id
              JOIN {DOC_T}   d ON d.doc_id   = c.doc_id
              WHERE NOT (c.chunk_id = ANY(%(x)s))
              GROUP BY c.chunk_id, c.doc_id, c.chunk_index, c.content, d.title
            ),
            ranked AS (SELECT *, row_number() OVER (PARTITION BY doc_id
                                                    ORDER BY graph_score DESC) AS rn FROM scored)
            SELECT * FROM ranked WHERE rn <= 2
            ORDER BY graph_score DESC, ent_hits DESC LIMIT %(l)s
        """, {"k": keys, "w": ws, "x": list(exclude) or [-1],
              "l": LOCAL_MAX_CHUNKS if limit is None else limit})
        return [dict(r) for r in cur.fetchall()]


LOCAL_RULES = f"""You are a business analyst. Answer using ONLY the material below.

Rules:
1. Cite source passages by number, e.g. [2]. Cite graph material as [graph]. No citation, no claim.
2. RELATIONSHIPS are extracted facts about how things connect. Use them to explain HOW and WHY,
   and to link evidence that appears in different passages.
3. If several entities or values legitimately match, list ALL of them with attribution.
   Never collapse distinct figures, and never present an aggregate as an individual's value.
4. Quote figures exactly as written — same units, scale, currency and precision.
5. If the material does not contain the answer, reply exactly: "{REFUSAL}"
6. Lead with the answer, then the supporting detail."""


def local_search(question, verbose=False):
    """Entity-anchored GraphRAG. For questions about specific things."""
    T, timings = time.perf_counter, {}
    t0 = T(); qv = embed_query(question); timings["embed_ms"] = (T() - t0) * 1000

    t0 = T(); seed = retrieve(question, qv=qv, top_k=max(LOCAL_SEED_CHUNKS, LOCAL_MAX_CHUNKS))
    timings["search_ms"] = (T() - t0) * 1000
    if not seed or (min(h["cosine_distance"] for h in seed) > MAX_COSINE_DISTANCE
                    and all(h["txt_rank"] is None for h in seed)):
        return {"answer": REFUSAL, "sources": [], "entities": [], "relations": [],
                "communities": [], "timings": timings, "refused": True}

    t0 = T()
    seed_ents = entities_in_chunks([h["chunk_id"] for h in seed[:LOCAL_SEED_CHUNKS]])
    expanded  = expand([e["entity_key"] for e in seed_ents])
    rels      = relations_between([e["entity_key"] for e in expanded])
    extra     = chunks_for_entities(expanded, exclude={h["chunk_id"] for h in seed})
    timings["graph_ms"] = (T() - t0) * 1000

    for e in extra:
        e.update({"cosine_distance": 1.0, "txt_rank": None, "via_graph": True})
    for h in seed:
        h["via_graph"] = False
    hits = (seed + extra)[:LOCAL_MAX_CHUNKS]

    cids = sorted({e["community"] for e in expanded if e.get("community") is not None})
    comms = []
    if cids:
        with db(dict_rows=True) as cur:
            cur.execute(f"SELECT community_id, title, summary FROM {COM_T} "
                        f"WHERE community_id = ANY(%s) AND summary IS NOT NULL "
                        f"ORDER BY rating DESC NULLS LAST LIMIT 3", (cids,))
            comms = [dict(r) for r in cur.fetchall()]

    ent_block  = "\n".join(f"  - {e['name']} ({e['entity_type']}): {e['description'][:280]}"
                            for e in expanded[:LOCAL_MAX_ENTITIES]) or "  (none)"
    rel_block  = "\n".join(f"  - {r['src']} ↔ {r['dst']}: {r['description'][:220]}"
                            for r in rels) or "  (none)"
    comm_block = "\n".join(f"  - {c['title']}: {(c['summary'] or '')[:600]}"
                            for c in comms) or "  (none)"
    src_block  = "\n\n".join(f"[{i}] {h['title']} (doc_id: {h['doc_id']})\n{h['content']}"
                              for i, h in enumerate(hits, 1))

    prompt = (f"{LOCAL_RULES}\n\nENTITIES:\n{ent_block}\n\nRELATIONSHIPS:\n{rel_block}\n\n"
              f"COMMUNITY CONTEXT:\n{comm_block}\n\nSOURCES:\n{src_block}\n\n"
              f"QUESTION: {question}")
    if verbose:
        print(prompt[:3000], "\n…\n")

    t0 = T()
    answer = gen(ANSWER_MODEL, prompt, max_tokens=4096, label="local answer").strip()
    timings["generate_ms"] = (T() - t0) * 1000
    return {"answer": answer,
            "sources": [{"n": i, "doc_id": h["doc_id"], "title": h["title"],
                         "via_graph": h.get("via_graph", False)} for i, h in enumerate(hits, 1)],
            "entities": expanded, "relations": rels, "communities": comms,
            "timings": timings, "refused": answer.startswith(REFUSAL)}

print("Local search ready — local_search(\"...\")")

## 8 — Global search

For questions about the corpus *as a whole*: themes, patterns, recurring risks. No chunk
contains the answer, so no amount of retrieval will find it.

The algorithm is **map-reduce over community summaries**:

1. **Map** — every community summary (above `GLOBAL_MIN_RATING`) is shown to the model in
   batches. For each batch it returns key points, each with a 0–100 score for how much that
   point helps answer *this* question.
2. **Reduce** — the highest-scoring points from all batches are combined into one answer.

This is why the indexing cost is worth paying: the expensive LLM pass over the corpus happened
once, at build time, and every global question now reads summaries instead of documents.

In [ ]:
MAP_PROMPT = """You are answering a question using summaries of communities in a knowledge graph.

For the QUESTION below, extract the key points from these COMMUNITY REPORTS that help answer it.

Score each point 0-100 for how directly and importantly it helps answer THIS question.
If a report is irrelevant, do not produce a point for it. Returning an empty list is correct
and expected when nothing is relevant.

Return RAW JSON only:
{{"points": [{{"description": "<a specific, self-contained statement>", "score": 85}}]}}

QUESTION: {question}

COMMUNITY REPORTS:
{reports}"""

REDUCE_PROMPT = """You are a business analyst writing a final answer.

Below are key points gathered from across the whole document corpus, each with an importance
score. Synthesise them into one coherent answer to the QUESTION.

Rules:
- Use ONLY these points. Add nothing from outside them.
- Organise by theme, not by score. Lead with what matters most.
- Where points conflict, say so rather than picking one silently.
- If the points do not answer the question, say exactly: "{refusal}"
- Be specific. Name names and quote figures as given.

QUESTION: {question}

KEY POINTS:
{points}"""


def global_search(question, min_rating=None, verbose=True):
    """Map-reduce over community summaries. For corpus-wide questions."""
    min_rating = GLOBAL_MIN_RATING if min_rating is None else min_rating
    T, timings = time.perf_counter, {}
    with db(dict_rows=True) as cur:
        cur.execute(f"""SELECT community_id, title, summary, rating, size FROM {COM_T}
                        WHERE summary IS NOT NULL AND COALESCE(rating, 0) >= %s
                        ORDER BY rating DESC NULLS LAST, size DESC""", (min_rating,))
        comms = [dict(r) for r in cur.fetchall()]
    if not comms:
        return {"answer": "No community summaries yet — run detect_communities() and "
                          "summarize_communities() first.", "points": [], "timings": timings}

    batches = [comms[i:i + GLOBAL_BATCH_SIZE] for i in range(0, len(comms), GLOBAL_BATCH_SIZE)]
    if verbose:
        print(f"Map: {len(comms)} communities in {len(batches)} batch(es)…")

    def _map(batch):
        reports = "\n\n".join(
            f"--- Community {c['community_id']}: {c['title']} (rating {c['rating']}) ---\n"
            f"{c['summary']}" for c in batch)
        try:
            data = parse_json(gen(ANSWER_MODEL,
                                  MAP_PROMPT.format(question=question, reports=reports),
                                  max_tokens=4096, json_out=True, label="global map"))
            return [p for p in (data.get("points") or [])
                    if isinstance(p, dict) and p.get("description")]
        except Exception as e:
            print(f"  ⚠️  map batch failed: {type(e).__name__}: {str(e)[:110]}")
            return []

    t0 = T()
    with ThreadPoolExecutor(max_workers=EXTRACT_WORKERS) as pool:
        points = [p for batch in pool.map(_map, batches) for p in batch]
    timings["map_ms"] = (T() - t0) * 1000

    def _score(p):
        try:
            return float(p.get("score") or 0)
        except (TypeError, ValueError):
            return 0.0
    points.sort(key=_score, reverse=True)
    top = points[:GLOBAL_TOP_POINTS]
    if verbose:
        print(f"   {len(points)} key point(s), keeping top {len(top)}")
    if not top:
        return {"answer": REFUSAL, "points": [], "timings": timings, "refused": True}

    t0 = T()
    answer = gen(ANSWER_MODEL,
                 REDUCE_PROMPT.format(question=question, refusal=REFUSAL,
                                      points="\n".join(f"  - ({_score(p):.0f}) {p['description']}"
                                                        for p in top)),
                 max_tokens=4096, label="global reduce").strip()
    timings["reduce_ms"] = (T() - t0) * 1000
    return {"answer": answer, "points": top, "communities_read": len(comms),
            "timings": timings, "refused": answer.startswith(REFUSAL)}

print("Global search ready — global_search(\"...\")")

## 9 — `ask()` — picking local or global

One cheap model call decides which search a question needs. The distinction is genuinely
simple, so this rarely gets it wrong:

- **local** — about specific named things: *"what happens if Tidewater's certificate lapses?"*
- **global** — about the corpus as a whole: *"what are the recurring compliance risks?"*

Following the lesson from your other notebook: if the router fails, it says so **loudly** and
falls back to local, rather than printing a routing decision that was actually a crash.

In [ ]:
ROUTE_PROMPT = """Decide which search answers this question about a document corpus.

"local"  = about specific named entities, facts, events or documents.
           Examples: "what are Globex's payment terms?", "why did bid 4471 fail?"
"global" = about the corpus AS A WHOLE — themes, patterns, trends, "what are the main…",
           "across all…", "what risks recur…", anything needing a view of everything.

QUESTION: {question}

Return RAW JSON only: {{"mode": "local" | "global", "reason": "<one short sentence>"}}"""


def route(question):
    try:
        data = parse_json(gen(ANSWER_MODEL, ROUTE_PROMPT.format(question=question),
                              max_tokens=512, json_out=True, label="router"))
        mode = str(data.get("mode", "")).lower()
        if mode not in ("local", "global"):
            raise ValueError(f"unexpected mode {mode!r}")
        return {"mode": mode, "reason": data.get("reason", "")}
    except Exception as e:
        print("⚠️  ROUTER FAILED — the answer below is a fallback, not a routing decision:")
        print(f"    {type(e).__name__}: {str(e)[:300]}")
        return {"mode": "local", "reason": f"Router unavailable ({type(e).__name__})."}


def ask(question, mode="auto", verbose=True, show=True):
    """The one entry point.  mode = "auto" | "local" | "global"."""
    if mode == "auto":
        d = route(question); mode = d["mode"]
        if verbose:
            print(f"{'🔍' if mode == 'local' else '🌍'} {mode.upper()} — {d['reason']}")
    elif verbose:
        print(f"➡️  forced to {mode.upper()}")

    result = local_search(question) if mode == "local" else global_search(question, verbose=verbose)
    if show:
        show_result(result, mode)
    return result


def show_result(r, mode="local"):
    print("\n" + r["answer"])
    if mode == "local":
        if r.get("entities"):
            by_hop = {}
            for e in r["entities"]:
                by_hop.setdefault(e["hop"], []).append(e["name"])
            print("\nEntities used:")
            for hop in sorted(by_hop):
                names = ", ".join(by_hop[hop][:6])
                more = f" (+{len(by_hop[hop]) - 6})" if len(by_hop[hop]) > 6 else ""
                print(f"  {hop} hop: {names}{more}")
        if r.get("relations"):
            print(f"\nRelationships given to the model ({len(r['relations'])}):")
            for x in r["relations"][:6]:
                print(f"  {x['src'][:24]} ↔ {x['dst'][:24]}: {x['description'][:70]}")
        if r.get("communities"):
            print("\nCommunity context:", ", ".join(c["title"] for c in r["communities"]))
        print("\nSources:")
        for s in r["sources"]:
            print(f"  [{s['n']}] {s['doc_id']}"
                  + ("  ← pulled in by the graph" if s["via_graph"] else ""))
    else:
        print(f"\nRead {r.get('communities_read', 0)} community summaries, "
              f"{len(r.get('points', []))} key points used.")
        for p in r.get("points", [])[:5]:
            print(f"  ({p.get('score')}) {p['description'][:100]}")
    print("  timings:", {k: round(v) for k, v in r["timings"].items()}, "ms")

print("✅ ask() ready.  ask(\"...\")  |  ask(\"...\", mode=\"local\")  |  mode=\"global\"")

## 10 — Build it, then ask

Run these in order the first time. Steps 2 and 4 cost money; steps 1 and 3 do not.

Extraction is resumable — if `EXTRACT_MAX_CHUNKS` stopped it early, raise the cap and call
`extract_graph()` again. It picks up where it left off.

In [ ]:
# ---- 1. get text in (uncomment ONE) --------------------------------------------------
# import_from_rag_tables()                     # reuse the other notebook's corpus
# add_document("doc-1", "Title", "Full text…", {"category": "Contract"})
# ingest()

# ---- 2. extract the graph (💸 one LLM call per chunk) --------------------------------
# extract_graph()

# ---- 3. clean up duplicate entities --------------------------------------------------
# dupes = find_duplicate_entities()            # dry run: read this list first
# merge_entities(dupes, apply=True)            # then commit

# ---- 4. communities (💸 one LLM call per community) ----------------------------------
# detect_communities()
# summarize_communities()

print("Uncomment the steps above and run them in order.")

In [ ]:
# Health check — where are you in the build?
def status():
    q = {"documents": DOC_T, "chunks": CHUNK_T, "entities": ENT_T,
         "relationships": REL_T, "mentions": MEN_T, "communities": COM_T}
    with db(dict_rows=True) as cur:
        for label, t in q.items():
            cur.execute(f"SELECT count(*) AS n FROM {t}")
            print(f"  {label:<16} {cur.fetchone()['n']:>7,}")
        cur.execute(f"SELECT count(*) AS n FROM {CHUNK_T} WHERE NOT extracted")
        pending = cur.fetchone()["n"]
        cur.execute(f"SELECT count(*) AS n FROM {COM_T} WHERE summary IS NULL")
        unsummarised = cur.fetchone()["n"]
    if pending:      print(f"  ⚠️  {pending} chunk(s) not yet extracted — run extract_graph()")
    if unsummarised: print(f"  ⚠️  {unsummarised} community(ies) unsummarised — run summarize_communities()")
    if not pending and not unsummarised:
        print("  ✅ index is complete")

status()

In [ ]:
# LOCAL — a question about specific things.
ask("What happens to the Valleverde contract if their cold-chain certification lapses?")

In [ ]:
# GLOBAL — a question no single chunk can answer.
# This is the capability your FK-derived notebook does not have.
ask("What are the recurring compliance and delivery risks across all suppliers?")

In [ ]:
# Inspect the graph by hand.
def find_entity(text, limit=15):
    with db(dict_rows=True) as cur:
        cur.execute(f"""SELECT entity_key, name, entity_type, degree, community
                        FROM {ENT_T} WHERE name ILIKE %s ORDER BY degree DESC LIMIT %s""",
                    (f"%{text}%", limit))
        rows = [dict(r) for r in cur.fetchall()]
    for r in rows:
        print(f"  {r['entity_type']:<14} {r['name'][:38]:<38} deg={r['degree']:<4} "
              f"community={r['community']}")
    return rows

def neighbours(entity_key, hops=1):
    rows = expand([entity_key], hops=hops)
    for r in rows:
        if r["hop"]:
            print(f"  {r['hop']} hop  {r['entity_type']:<14} {r['name'][:44]}")
    return rows

def communities(top=15):
    with db(dict_rows=True) as cur:
        cur.execute(f"""SELECT community_id, size, rating, title FROM {COM_T}
                        ORDER BY rating DESC NULLS LAST LIMIT %s""", (top,))
        for r in cur.fetchall():
            print(f"  #{r['community_id']:<3} size={r['size']:<4} rating={r['rating']} "
                  f"{r['title']}")

# find_entity("Tidewater")
# neighbours("tidewater frozen holdings", hops=2)
# communities()
print("Inspectors ready — find_entity(), neighbours(), communities()")

## 11 — What this costs, and when it is worth it

**Indexing is the bill.** One extraction call per chunk (plus one gleaning), then one summary
call per community. A 500-chunk corpus is roughly 1,000 extraction calls and perhaps 30 summary
calls. Querying afterwards is ordinary RAG pricing.

**Re-indexing is not free either.** Change a document and its chunks need re-extracting; change
enough of them and the communities shift, so summaries need regenerating. Budget for this if
your corpus moves.

**Use this notebook when** your relationships exist only in prose — contracts, emails, reports,
meeting notes — and nothing in your schema knows about them. And when you need corpus-wide
questions answered, which is the thing only global search can do.

**Use your other notebook when** the relationships are already declared as foreign keys or
document metadata. Deriving them is free, deterministic and exactly correct; paying an LLM to
guess at something the database already knows is slower, costlier and occasionally wrong.

**They are not mutually exclusive.** The strongest setup is both: FK-derived edges as the
trustworthy skeleton, LLM-extracted edges for what only the prose knows. They write to different
tables here, so nothing stops you from merging them later — and if you do, keep the `source`
column so you can always tell a fact from a guess.